Appendix E: Parameter-efficient Finetuning with LoRA

In [ ]:
# 中文注释：检查各依赖库的安装版本，确保运行环境满足教程要求
from importlib.metadata import version

pkgs = ["matplotlib",
        "numpy",
        "tiktoken",
        "torch",
        "tensorflow", # For OpenAI's pretrained weights
        "pandas"      # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# import urllib
# 中文注释：使用 requests 库下载数据集压缩包（原书使用 urllib，因部分 VPN 环境下常报错而改用 requests，见下方说明）
import requests
from pathlib import Path
import pandas as pd
# 中文注释：从本地 previous_chapters.py 导入第6章实现的：数据集下载解压、类别均衡采样、随机切分函数
from previous_chapters import (
    download_and_unzip_spam_data,
    create_balanced_dataset,
    random_split
)
# If the `previous_chapters.py` file is not available locally,
# you can import it from the `llms-from-scratch` PyPI package.
# For details, see: https://github.com/rasbt/LLMs-from-scratch/tree/main/pkg
# E.g.,
# from llms_from_scratch.ch06 import (
#     download_and_unzip_spam_data,
#     create_balanced_dataset,
#     random_split
# )



# 中文注释：设置垃圾短信数据集的下载地址及本地解压路径
url = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
zip_path = "sms_spam_collection.zip"
extracted_path = "sms_spam_collection"
data_file_path = Path(extracted_path) / "SMSSpamCollection.tsv"


# 中文注释：优先尝试主 URL 下载，若网络请求失败（超时/连接错误）则自动切换到备用镜像地址重试
try:
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)
except (requests.exceptions.RequestException, TimeoutError) as e:
    print(f"Primary URL failed: {e}. Trying backup URL...")
    url = "https://f001.backblazeb2.com/file/LLMs-from-scratch/sms%2Bspam%2Bcollection.zip"
    download_and_unzip_spam_data(url, zip_path, extracted_path, data_file_path)

# The book originally used
# except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError) as e:
# in the code above.
# However, some VPN users reported issues with `urllib`, so the code was updated
# to use `requests` instead

# 中文注释：读取制表符分隔的原始数据（标签 + 短信文本）
df = pd.read_csv(data_file_path, sep="\t", header=None, names=["Label", "Text"])
# 中文注释：对 ham（正常短信）做下采样，使其数量与 spam（垃圾短信）一致，得到类别均衡的数据集
balanced_df = create_balanced_dataset(df)
# 中文注释：将文本标签映射为数值标签（ham=0, spam=1），便于后续分类训练
balanced_df["Label"] = balanced_df["Label"].map({"ham": 0, "spam": 1})

# 中文注释：按 70%/10%/20% 比例随机切分为训练集、验证集、测试集
train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# 中文注释：分别保存为 CSV 文件，供后续 SpamDataset 读取使用
train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [ ]:
# 中文注释：构建 SpamDataset 数据集与 DataLoader，用于垃圾短信二分类微调
import torch
import tiktoken
from previous_chapters import SpamDataset


# 中文注释：使用 GPT-2 的 BPE 分词器对文本进行编码
tokenizer = tiktoken.get_encoding("gpt2")
# 中文注释：训练集不指定 max_length（传 None），由 SpamDataset 内部自动取训练集中最长样本的 token 长度作为统一长度
train_dataset = SpamDataset("train.csv", max_length=None, tokenizer=tokenizer)
# 中文注释：验证集/测试集复用训练集算出的 max_length，保证三者输入维度一致，避免尺寸不匹配
val_dataset = SpamDataset("validation.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
test_dataset = SpamDataset("test.csv", max_length=train_dataset.max_length, tokenizer=tokenizer)
from torch.utils.data import DataLoader

# 中文注释：设置 DataLoader 的工作进程数与批大小
num_workers = 0
batch_size = 8

# 中文注释：固定随机种子，保证 shuffle 顺序可复现
torch.manual_seed(123)

# 中文注释：训练集 DataLoader 打乱顺序，并丢弃最后不足一个 batch_size 的样本（drop_last=True），避免 batch 大小不一致
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

# 中文注释：验证集/测试集不打乱顺序，也不丢弃最后不完整的 batch，保证评估覆盖全部样本
val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [ ]:
# 中文注释：遍历一次训练 DataLoader（这里只是为了取出最后一个 batch，检查其形状；pass 表示不做其他处理）
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

# 中文注释：打印最后一个 batch 的输入 token 序列形状 (batch_size, seq_len) 及标签形状 (batch_size,)
print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

In [ ]:
# 中文注释：打印训练/验证/测试集分别包含多少个 batch，用于快速核对数据量是否符合预期
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

E.3 Initializing the model

In [ ]:
# 中文注释：下载并加载官方 OpenAI 预训练的 GPT-2 权重，构建对应结构的自定义 GPTModel
from gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel, load_weights_into_gpt
# Alternatively:
# from llms_from_scratch.ch04 import GPTModel
# from llms_from_scratch.ch05 import load_weights_into_gpt



# 中文注释：选择要加载的 GPT-2 规模，以及用于后续生成文本验证的提示词
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

# 中文注释：GPT-2 通用基础配置（词表大小、上下文长度、dropout 比例、qkv 是否带偏置）
BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

# 中文注释：不同规模 GPT-2 对应的嵌入维度、层数、注意力头数配置
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# 中文注释：将所选规模的专属配置合并进基础配置
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

# 中文注释：从模型名称中提取参数量标识（如 '124M'），用于定位下载文件
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
# 中文注释：下载（若本地已有则跳过）并加载 GPT-2 预训练权重及其配置
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

# 中文注释：实例化自定义 GPTModel 结构，并将下载到的官方权重逐层加载进去
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
# 中文注释：切换为 eval 模式（关闭 dropout 等训练专属行为），准备做推理验证
model.eval();

In [ ]:
# 中文注释：用一段示例文本做自回归生成，验证预训练权重加载是否正确（微调前的健全性检查）
from previous_chapters import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)


text_1 = "Every effort moves you"

# 中文注释：以贪心方式自回归生成 15 个新 token
token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

In [ ]:
# 中文注释：将语言模型改造为二分类器，并选择计算设备
torch.manual_seed(123)

# 中文注释：垃圾短信分类任务只有两类：ham（正常）/ spam（垃圾）
num_classes = 2
# 中文注释：把原来输出词表大小 logits 的语言模型头，替换为输出 2 个类别 logits 的线性分类头
model.out_head = torch.nn.Linear(in_features=768, out_features=num_classes)
# 中文注释：按优先级选择可用的计算设备：CUDA > MPS（Apple Silicon，且要求 PyTorch>=2.9 才有稳定结果） > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    # Use PyTorch 2.9 or newer for stable mps results
    major, minor = map(int, torch.__version__.split(".")[:2])
    if (major, minor) >= (2, 9):
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
else:
    device = torch.device("cpu")

print("Device:", device)

# 中文注释：将模型搬到目标设备；nn.Module 的 .to() 是原地操作，不需要重新赋值
model.to(device);  # no assignment model = model.to(device) necessary for nn.Module classes

In [ ]:
# 中文注释：导入准确率计算工具函数
from previous_chapters import calc_accuracy_loader
# Alternatively:
# from llms_from_scratch.ch06 import calc_accuracy_loader



# 中文注释：在替换分类头之后、注入 LoRA 之前，先评估一次基线准确率（此时除输出头外其余参数均为预训练权重且仍可训练）
torch.manual_seed(123)
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

E.4 Parameter-efficient finetuning with LoRA

In [ ]:
# 中文注释：实现 LoRA（Low-Rank Adaptation，低秩适应）核心模块
import math

# 中文注释：LoRALayer 用一对低秩矩阵 A、B 近似表示对原始权重矩阵的增量更新 ΔW ≈ A @ B
class LoRALayer(torch.nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        # 中文注释：A 的形状为 (in_dim, rank)，将输入从 in_dim 维降到 rank 维（低秩瓶颈）
        self.A = torch.nn.Parameter(torch.empty(in_dim, rank))
        # 中文注释：对 A 使用 kaiming 均匀初始化（与 nn.Linear 默认权重初始化方式相近），使其初始输出具有合理量级
        torch.nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))  # similar to standard weight initialization
        # 中文注释：B 的形状为 (rank, out_dim)，将 rank 维重新映射回 out_dim 维；初始化为全零
        self.B = torch.nn.Parameter(torch.zeros(rank, out_dim))
        # 中文注释：B 初始化为 0，使得训练开始时 A @ B = 0，LoRA 分支不改变原模型输出，保证微调起点与预训练模型一致
        self.alpha = alpha
        # 中文注释：alpha 为缩放系数，rank 为低秩维度，二者共同决定 LoRA 更新的整体幅度
        self.rank = rank

    def forward(self, x):
        # Note: The original chapter didn't include the scaling by self.rank
        # This scaling is not necessary, but it's more canonical and convenient
        # as this lets us compare runs across different ranks without retuning learning rates
        # 中文注释：x @ A 将输入投影到 rank 维，再 @ B 投影回 out_dim 维，得到低秩近似的权重增量作用效果；
        # 乘以 alpha/rank 是缩放因子，alpha 越大代表 LoRA 更新对最终输出影响越大，除以 rank 便于跨不同 rank 取值比较而不用重调学习率
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x

In [ ]:
# 中文注释：LinearWithLoRA 用「冻结的原始线性层 + 并联的 LoRA 低秩分支」替换原来的 nn.Linear
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        # 中文注释：保留原始（预训练）线性层，后续会被冻结，不参与梯度更新
        self.linear = linear
        # 中文注释：为该线性层新增一个 LoRA 分支，输入/输出维度与原线性层保持一致
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # 中文注释：前向传播 = 原始线性层输出 + LoRA 分支输出，等价于用低秩增量 ΔW 对原权重做旁路修正
        return self.linear(x) + self.lora(x)

In [ ]:
# 中文注释：递归遍历模型的所有子模块，把每一个 nn.Linear 都替换为 LinearWithLoRA
def replace_linear_with_lora(model, rank, alpha):
    # 中文注释：named_children() 只返回直接子模块，因此需要递归才能触达深层嵌套的 Linear 层（如注意力/前馈网络内部）
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            # Replace the Linear layer with LinearWithLoRA
            setattr(model, name, LinearWithLoRA(module, rank, alpha))
        else:
            # Recursively apply the same function to child modules
            replace_linear_with_lora(module, rank, alpha)

In [ ]:
# 中文注释：冻结基础模型的所有参数，为后续只训练新增的 LoRA 参数做准备
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters before: {total_params:,}")

# 中文注释：遍历模型全部参数，将 requires_grad 设为 False，使反向传播时不再计算/更新这些参数的梯度
for param in model.parameters():
    param.requires_grad = False

# 中文注释：冻结后可训练参数应变为 0（此时还未插入 LoRA 层）
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters after: {total_params:,}")

In [ ]:
# 中文注释：将 LoRA 注入到模型的所有线性层中，rank=16（低秩维度），alpha=16（缩放系数）
replace_linear_with_lora(model, rank=16, alpha=16)

# 中文注释：新插入的 LoRALayer 中的 A、B 是新建的 nn.Parameter，默认 requires_grad=True，不受之前冻结循环影响，
# 因此这里统计出的可训练参数数量就是全部 LoRA 参数量（远小于原模型全部参数量）
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable LoRA parameters: {total_params:,}")

In [ ]:
# 中文注释：把包含新增 LoRA 参数的模型重新搬到目标设备，并打印模型结构以确认 Linear 层已被 LinearWithLoRA 替换
model.to(device)

print(model)

In [ ]:
# 中文注释：刚插入 LoRA、尚未开始训练时，再次评估准确率
torch.manual_seed(123)
# 中文注释：由于 LoRALayer 中 B 初始化为全零，此时 LoRA 分支输出恒为 0，模型效果应与插入 LoRA 之前完全一致
train_accuracy = calc_accuracy_loader(train_loader, model, device, num_batches=10)
val_accuracy = calc_accuracy_loader(val_loader, model, device, num_batches=10)
test_accuracy = calc_accuracy_loader(test_loader, model, device, num_batches=10)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")

In [ ]:
# 中文注释：开始对模型进行 LoRA 微调训练
import time
from previous_chapters import train_classifier_simple
# Alternatively:
# from llms_from_scratch.ch06 import train_classifier_simple


start_time = time.time()

torch.manual_seed(123)

# 中文注释：优化器传入 model.parameters() 虽然包含全部参数，但只有 requires_grad=True 的 LoRA A/B 矩阵才会被实际更新；
# 被冻结的原始权重梯度为 None，不会产生参数更新
optimizer = torch.optim.AdamW(model.parameters(), lr=8e-4, weight_decay=0.1)

num_epochs = 5
# 中文注释：调用第6章实现的标准分类训练循环，每 50 步做一次验证（eval_freq），每次验证用 5 个 batch（eval_iter），共训练 5 个 epoch
train_losses, val_losses, train_accs, val_accs, examples_seen = train_classifier_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=50, eval_iter=5,
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

In [ ]:
# 中文注释：绘制训练/验证损失曲线
from previous_chapters import plot_values
# Alternatively:
# from llms_from_scratch.ch06 import plot_values

# 中文注释：构造与记录点数量一致的 x 轴刻度（分别按 epoch 数和按累计样本数两种口径）
epochs_tensor = torch.linspace(0, num_epochs, len(train_losses))
examples_seen_tensor = torch.linspace(0, examples_seen, len(train_losses))

plot_values(epochs_tensor, examples_seen_tensor, train_losses, val_losses, label="loss")

In [ ]:
# 中文注释：LoRA 微调完成后，在完整的训练/验证/测试集（不限制 num_batches）上评估最终准确率
train_accuracy = calc_accuracy_loader(train_loader, model, device)
val_accuracy = calc_accuracy_loader(val_loader, model, device)
test_accuracy = calc_accuracy_loader(test_loader, model, device)

print(f"Training accuracy: {train_accuracy*100:.2f}%")
print(f"Validation accuracy: {val_accuracy*100:.2f}%")
print(f"Test accuracy: {test_accuracy*100:.2f}%")